In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np
import os

In [ ]:
project_path = '/content/drive/MyDrive/Brazilian e_commerce'

raw_path = os.path.join(project_path, 'Raw')
processed_path = os.path.join(project_path, 'Processed')

print("Raw folder exists:", os.path.exists(raw_path))
print("Processed folder exists:", os.path.exists(processed_path))

Raw folder exists: True
Processed folder exists: True


In [ ]:
# Loading all nine datasets

customers = pd.read_csv(os.path.join(raw_path, 'olist_customers_dataset.csv'))

geolocation = pd.read_csv(os.path.join(raw_path, 'olist_geolocation_dataset.csv'))

order_items = pd.read_csv(os.path.join(raw_path, 'olist_order_items_dataset.csv'))

order_payments = pd.read_csv(os.path.join(raw_path, 'olist_order_payments_dataset.csv'))

order_reviews = pd.read_csv(os.path.join(raw_path, 'olist_order_reviews_dataset.csv'))

orders = pd.read_csv(os.path.join(raw_path, 'olist_orders_dataset.csv'))

products = pd.read_csv(os.path.join(raw_path, 'olist_products_dataset.csv'))

sellers = pd.read_csv(os.path.join(raw_path, 'olist_sellers_dataset.csv'))

category_translation = pd.read_csv(os.path.join(raw_path, 'product_category_name_translation.csv'))

print("All nine datasets loaded successfully!")

All nine datasets loaded successfully!


# **Create working copies**

In [ ]:
customers_clean = customers.copy()
geolocation_clean = geolocation.copy()
order_items_clean = order_items.copy()
order_payments_clean = order_payments.copy()
order_reviews_clean = order_reviews.copy()
orders_clean = orders.copy()
products_clean = products.copy()
sellers_clean = sellers.copy()
category_translation_clean = category_translation.copy()

print("Working copies created successfully!")

Working copies created successfully!


# **Handling Missing Values**

# 1.1)Handle missing values in Order Reviews

In [ ]:
print("Before handling missing values:")

print(order_reviews_clean[
    ['review_comment_title', 'review_comment_message']
].isnull().sum())

Before handling missing values:
review_comment_title      87656
review_comment_message    58247
dtype: int64


# 1.2)Handle missing values in Orders

In [ ]:
#Check missing delivery dates against order status
missing_delivery = orders_clean[
    orders_clean['order_delivered_customer_date'].isnull()
]

print("Orders with missing customer delivery date:")
print("Total:", len(missing_delivery))

display(
    missing_delivery['order_status']
    .value_counts()
    .to_frame('Number of Orders')
)

Orders with missing customer delivery date:
Total: 2965


,Number of Orders
order_status,
shipped,1107
canceled,619
unavailable,609
invoiced,314
processing,301
delivered,8
created,5
approved,2


In [ ]:
#Investigate the other missing dates
date_columns = [
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date'
]

for column in date_columns:

    print("\nMissing values in:", column)

    missing_orders = orders_clean[
        orders_clean[column].isnull()
    ]

    display(
        missing_orders['order_status']
        .value_counts()
        .to_frame('Number of Orders')
    )


Missing values in: order_approved_at


,Number of Orders
order_status,
canceled,141
delivered,14
created,5



Missing values in: order_delivered_carrier_date


,Number of Orders
order_status,
unavailable,609
canceled,550
invoiced,314
processing,301
created,5
approved,2
delivered,2



Missing values in: order_delivered_customer_date


,Number of Orders
order_status,
shipped,1107
canceled,619
unavailable,609
invoiced,314
processing,301
delivered,8
created,5
approved,2


# 1.3)Handle missing values in Products

In [ ]:
#Replace missing product categories with unknown
products_clean['product_category_name'] = (
    products_clean['product_category_name']
    .fillna('unknown')
)

In [ ]:
print(
    "Missing product categories after handling:",
    products_clean['product_category_name'].isnull().sum()
)

print(
    "Products classified as unknown:",
    (products_clean['product_category_name'] == 'unknown').sum()
)

Missing product categories after handling: 0
Products classified as unknown: 610


# 1.4)Verify missing values after our changes

In [ ]:
datasets_clean = {
    'Customers': customers_clean,
    'Geolocation': geolocation_clean,
    'Order Items': order_items_clean,
    'Order Payments': order_payments_clean,
    'Order Reviews': order_reviews_clean,
    'Orders': orders_clean,
    'Products': products_clean,
    'Sellers': sellers_clean,
    'Category Translation': category_translation_clean
}

remaining_missing = []

for name, df in datasets_clean.items():

    for column in df.columns:

        missing_count = df[column].isnull().sum()

        if missing_count > 0:

            remaining_missing.append({
                'Dataset': name,
                'Column': column,
                'Missing Values': missing_count
            })

remaining_missing_df = pd.DataFrame(remaining_missing)

display(remaining_missing_df)

,Dataset,Column,Missing Values
0,Order Reviews,review_comment_title,87656
1,Order Reviews,review_comment_message,58247
2,Orders,order_approved_at,160
3,Orders,order_delivered_carrier_date,1783
4,Orders,order_delivered_customer_date,2965
5,Products,product_name_lenght,610
6,Products,product_description_lenght,610
7,Products,product_photos_qty,610
8,Products,product_weight_g,2
9,Products,product_length_cm,2


# **Handling Duplicate Records**

# 1.1)Investigate geolocation duplicates

In [ ]:
#Display the duplicate records
geo_duplicates = geolocation_clean[
    geolocation_clean.duplicated(keep=False)
]

print("Total rows involved in duplicates:", len(geo_duplicates))

display(geo_duplicates.head(10))

Total rows involved in duplicates: 390005


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
6,1047,-23.546273,-46.641225,sao paulo,SP
7,1013,-23.546923,-46.634264,sao paulo,SP
8,1029,-23.543769,-46.634278,sao paulo,SP
9,1011,-23.547640,-46.636032,sao paulo,SP
10,1013,-23.547325,-46.634184,sao paulo,SP
13,1012,-23.548946,-46.634671,sao paulo,SP
15,1046,-23.546081,-46.644820,sao paulo,SP


In [ ]:
#Check how many unique records remain
print("Original geolocation rows:", len(geolocation_clean))

print(
    "Unique geolocation rows:",
    len(geolocation_clean.drop_duplicates())
)

print(
    "Exact duplicate rows:",
    geolocation_clean.duplicated().sum()
)

Original geolocation rows: 1000163
Unique geolocation rows: 738332
Exact duplicate rows: 261831


# 1.2)Remove exact duplicates

In [ ]:
#Create a cleaned geolocation dataset
geolocation_clean = geolocation_clean.drop_duplicates()

print("Geolocation duplicates removed successfully!")

print("Rows after cleaning:", len(geolocation_clean))

print(
    "Remaining exact duplicates:",
    geolocation_clean.duplicated().sum()
)

Geolocation duplicates removed successfully!
Rows after cleaning: 738332
Remaining exact duplicates: 0


# 1.3)Verify other datasets

In [ ]:
#Check exact duplicates again
datasets_clean = {
    'Customers': customers_clean,
    'Geolocation': geolocation_clean,
    'Order Items': order_items_clean,
    'Order Payments': order_payments_clean,
    'Order Reviews': order_reviews_clean,
    'Orders': orders_clean,
    'Products': products_clean,
    'Sellers': sellers_clean,
    'Category Translation': category_translation_clean
}

for name, df in datasets_clean.items():

    print(
        name,
        ":",
        df.duplicated().sum(),
        "exact duplicate rows remaining"
    )

Customers : 0 exact duplicate rows remaining
Geolocation : 261831 exact duplicate rows remaining
Order Items : 0 exact duplicate rows remaining
Order Payments : 0 exact duplicate rows remaining
Order Reviews : 0 exact duplicate rows remaining
Orders : 0 exact duplicate rows remaining
Products : 0 exact duplicate rows remaining
Sellers : 0 exact duplicate rows remaining
Category Translation : 0 exact duplicate rows remaining


In [ ]:
#Investigate repeated postal-code prefixes
print(
    "Total geolocation rows:",
    len(geolocation_clean)
)

print(
    "Unique postal-code prefixes:",
    geolocation_clean['geolocation_zip_code_prefix'].nunique()
)

print(
    "Repeated postal-code prefixes:",
    geolocation_clean[
        'geolocation_zip_code_prefix'
    ].duplicated().sum()
)

Total geolocation rows: 1000163
Unique postal-code prefixes: 19015
Repeated postal-code prefixes: 981148


# 1.4)Create a duplicate-cleaning report

# **Data Cleaning**

# 1.1)checking current datatype

In [ ]:
print("ORDERS DATASET")
print(orders_clean[[
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]].dtypes)

print("\nORDER ITEMS DATASET")
print(order_items_clean['shipping_limit_date'].dtype)

print("\nORDER REVIEWS DATASET")
print(order_reviews_clean[[
    'review_creation_date',
    'review_answer_timestamp'
]].dtypes)

ORDERS DATASET
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object

ORDER ITEMS DATASET
object

ORDER REVIEWS DATASET
review_creation_date       object
review_answer_timestamp    object
dtype: object


# 1.2)converting date columns into datetime

In [ ]:
order_date_columns = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for column in order_date_columns:
    orders_clean[column] = pd.to_datetime(
        orders_clean[column],
        errors='coerce'
    )

print("Orders date columns converted successfully!")

Orders date columns converted successfully!


# 1.3)Convert the Order Items date column

In [ ]:
order_items_clean['shipping_limit_date'] = pd.to_datetime(
    order_items_clean['shipping_limit_date'],
    errors='coerce'
)

print("Order Items date column converted successfully!")

Order Items date column converted successfully!


# 1.4)Convert the Order Reviews date columns

In [ ]:
review_date_columns = [
    'review_creation_date',
    'review_answer_timestamp'
]

for column in review_date_columns:
    order_reviews_clean[column] = pd.to_datetime(
        order_reviews_clean[column],
        errors='coerce'
    )

print("Order Reviews date columns converted successfully!")

Order Reviews date columns converted successfully!


# 1.5)Check the new data types

In [ ]:
print("ORDERS")
print(orders_clean[order_date_columns].dtypes)

print("\nORDER ITEMS")
print(order_items_clean['shipping_limit_date'].dtype)

print("\nORDER REVIEWS")
print(order_reviews_clean[review_date_columns].dtypes)

ORDERS
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

ORDER ITEMS
datetime64[ns]

ORDER REVIEWS
review_creation_date       datetime64[ns]
review_answer_timestamp    datetime64[ns]
dtype: object


In [ ]:
date_datasets = {
    'Orders': (orders_clean, order_date_columns),
    'Order Items': (order_items_clean, ['shipping_limit_date']),
    'Order Reviews': (order_reviews_clean, review_date_columns)
}

for name, (df, columns) in date_datasets.items():

    print("\n==========", name, "==========")

    for column in columns:
        print(
            column,
            ":",
            df[column].isna().sum(),
            "missing or invalid dates"
        )


========== Orders ==========
order_purchase_timestamp : 0 missing or invalid dates
order_approved_at : 160 missing or invalid dates
order_delivered_carrier_date : 1783 missing or invalid dates
order_delivered_customer_date : 2965 missing or invalid dates
order_estimated_delivery_date : 0 missing or invalid dates

========== Order Items ==========
shipping_limit_date : 0 missing or invalid dates

========== Order Reviews ==========
review_creation_date : 0 missing or invalid dates
review_answer_timestamp : 0 missing or invalid dates


1.6)Correct other important data types

In [ ]:
# check numeric columns
numeric_columns = {
    'Order Items': (order_items_clean, ['price', 'freight_value']),
    'Order Payments': (
        order_payments_clean,
        ['payment_sequential', 'payment_installments', 'payment_value']
    ),
    'Order Reviews': (order_reviews_clean, ['review_score']),
    'Products': (
        products_clean,
        [
            'product_name_lenght',
            'product_description_lenght',
            'product_photos_qty',
            'product_weight_g',
            'product_length_cm',
            'product_height_cm',
            'product_width_cm'
        ]
    )
}

for name, (df, columns) in numeric_columns.items():

    print("\n", name)

    for column in columns:
        print(column, ":", df[column].dtype)


 Order Items
price : float64
freight_value : float64

 Order Payments
payment_sequential : int64
payment_installments : int64
payment_value : float64

 Order Reviews
review_score : int64

 Products
product_name_lenght : float64
product_description_lenght : float64
product_photos_qty : float64
product_weight_g : float64
product_length_cm : float64
product_height_cm : float64
product_width_cm : float64


# 1.7)Treat postal codes and IDs correctly

In [ ]:
customers_clean['customer_zip_code_prefix'] = (
    customers_clean['customer_zip_code_prefix'].astype('string')
)

sellers_clean['seller_zip_code_prefix'] = (
    sellers_clean['seller_zip_code_prefix'].astype('string')
)

geolocation_clean['geolocation_zip_code_prefix'] = (
    geolocation_clean['geolocation_zip_code_prefix'].astype('string')
)

print("Postal-code prefixes converted to string!")

Postal-code prefixes converted to string!


# 1.8)Create a data-type report

In [ ]:
dtype_reports = []

for name, df in datasets_clean.items():

    for column in df.columns:

        dtype_reports.append({
            'Dataset': name,
            'Column': column,
            'Data Type': str(df[column].dtype)
        })

dtype_report = pd.DataFrame(dtype_reports)

display(dtype_report)

,Dataset,Column,Data Type
0,Customers,customer_id,object
1,Customers,customer_unique_id,object
2,Customers,customer_zip_code_prefix,string
3,Customers,customer_city,object
4,Customers,customer_state,object
5,Geolocation,geolocation_zip_code_prefix,string
6,Geolocation,geolocation_lat,float64
7,Geolocation,geolocation_lng,float64
8,Geolocation,geolocation_city,object
9,Geolocation,geolocation_state,object


# **Standardizing Product Category Name**

# 1.1)Check the product categories before translation

In [ ]:
print("Number of unique product categories:")

print(
    products_clean['product_category_name'].nunique()
)

print("\nSample product categories:")

display(
    products_clean['product_category_name']
    .value_counts()
    .head(15)
)

Number of unique product categories:
74

Sample product categories:


,count
product_category_name,
cama_mesa_banho,3029
esporte_lazer,2867
moveis_decoracao,2657
beleza_saude,2444
utilidades_domesticas,2335
automotivo,1900
informatica_acessorios,1639
brinquedos,1411
relogios_presentes,1329


# 1.2)Merge the translation file with the product dataset

In [ ]:
products_clean = products_clean.merge(
    category_translation_clean,
    on='product_category_name',
    how='left'
)

print("Translation merge completed!")

Translation merge completed!


# 1.3)Create a standardized English category column

In [ ]:
products_clean['product_category_name_english'] = (
    products_clean['product_category_name_english']
    .fillna(products_clean['product_category_name'])
)

In [ ]:
display(
    products_clean[
        [
            'product_category_name',
            'product_category_name_english'
        ]
    ].drop_duplicates().head(20)
)

,product_category_name,product_category_name_english
0,perfumaria,perfumery
1,artes,art
2,esporte_lazer,sports_leisure
3,bebes,baby
4,utilidades_domesticas,housewares
5,instrumentos_musicais,musical_instruments
6,cool_stuff,cool_stuff
7,moveis_decoracao,furniture_decor
8,eletrodomesticos,home_appliances
9,brinquedos,toys


# 1.4)Check whether translations are missing

In [ ]:
print(
    "Missing standardized categories:",
    products_clean['product_category_name_english'].isnull().sum()
)

print(
    "Number of standardized categories:",
    products_clean['product_category_name_english'].nunique()
)

Missing standardized categories: 0
Number of standardized categories: 74


# 1.5)Check the translated category distribution

In [ ]:
display(
    products_clean['product_category_name_english']
    .value_counts()
    .head(15)
    .to_frame('Number of Products')
)

,Number of Products
product_category_name_english,
bed_bath_table,3029
sports_leisure,2867
furniture_decor,2657
health_beauty,2444
housewares,2335
auto,1900
computers_accessories,1639
toys,1411
watches_gifts,1329


# **Check invalid and inconsistent values**

In [ ]:
print(orders_clean['order_status'].value_counts())

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [ ]:
print("Missing order statuses:",
      orders_clean['order_status'].isnull().sum())

print("Unique order statuses:",
      orders_clean['order_status'].unique())

Missing order statuses: 0
Unique order statuses: ['delivered' 'invoiced' 'shipped' 'processing' 'unavailable' 'canceled'
 'created' 'approved']


In [ ]:
# check delivery days
invalid_delivery = orders_clean[
    (orders_clean['order_delivered_customer_date'] <
     orders_clean['order_purchase_timestamp'])
    &
    orders_clean['order_delivered_customer_date'].notna()
]

print("Orders delivered before purchase:", len(invalid_delivery))

display(invalid_delivery.head())

Orders delivered before purchase: 0


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date


In [ ]:
late_delivery = orders_clean[
    orders_clean['order_delivered_customer_date'] >
    orders_clean['order_estimated_delivery_date']
]

print("Orders delivered after estimated date:", len(late_delivery))

Orders delivered after estimated date: 7827


In [ ]:
# Check order item values
print("Negative prices:",
      (order_items_clean['price'] < 0).sum())

print("Zero prices:",
      (order_items_clean['price'] == 0).sum())

print("Missing prices:",
      order_items_clean['price'].isnull().sum())

Negative prices: 0
Zero prices: 0
Missing prices: 0


In [ ]:
print("Negative freight values:",
      (order_items_clean['freight_value'] < 0).sum())

print("Zero freight values:",
      (order_items_clean['freight_value'] == 0).sum())

Negative freight values: 0
Zero freight values: 383


In [ ]:
print("Negative payment values:",
      (order_payments_clean['payment_value'] < 0).sum())

print("Zero payment values:",
      (order_payments_clean['payment_value'] == 0).sum())

print("Invalid installment values:",
      (order_payments_clean['payment_installments'] < 1).sum())

Negative payment values: 0
Zero payment values: 9
Invalid installment values: 2


In [ ]:
print(order_payments_clean['payment_type'].value_counts())

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64


In [ ]:
print("Minimum review score:",
      order_reviews_clean['review_score'].min())

print("Maximum review score:",
      order_reviews_clean['review_score'].max())

print("Invalid review scores:",
      (~order_reviews_clean['review_score'].between(1, 5)).sum())

Minimum review score: 1
Maximum review score: 5
Invalid review scores: 0


In [ ]:
print(order_reviews_clean['review_score'].value_counts().sort_index())

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64


In [ ]:
measurement_columns = [
    'product_weight_g',
    'product_length_cm',
    'product_height_cm',
    'product_width_cm'
]

for column in measurement_columns:

    print("\n", column)

    print("Negative values:",
          (products_clean[column] < 0).sum())

    print("Zero values:",
          (products_clean[column] == 0).sum())

    print("Missing values:",
          products_clean[column].isnull().sum())


 product_weight_g
Negative values: 0
Zero values: 4
Missing values: 2

 product_length_cm
Negative values: 0
Zero values: 0
Missing values: 2

 product_height_cm
Negative values: 0
Zero values: 0
Missing values: 2

 product_width_cm
Negative values: 0
Zero values: 0
Missing values: 2


In [ ]:
print("Customer postal code lengths:")
print(customers_clean['customer_zip_code_prefix'].str.len().value_counts())

print("\nSeller postal code lengths:")
print(sellers_clean['seller_zip_code_prefix'].str.len().value_counts())

Customer postal code lengths:
customer_zip_code_prefix
5    75446
4    23995
Name: count, dtype: Int64

Seller postal code lengths:
seller_zip_code_prefix
5    2068
4    1027
Name: count, dtype: Int64


In [ ]:
print("Customer states:")
print(sorted(customers_clean['customer_state'].unique()))

print("\nSeller states:")
print(sorted(sellers_clean['seller_state'].unique()))

Customer states:
['AC', 'AL', 'AM', 'AP', 'BA', 'CE', 'DF', 'ES', 'GO', 'MA', 'MG', 'MS', 'MT', 'PA', 'PB', 'PE', 'PI', 'PR', 'RJ', 'RN', 'RO', 'RR', 'RS', 'SC', 'SE', 'SP', 'TO']

Seller states:
['AC', 'AM', 'BA', 'CE', 'DF', 'ES', 'GO', 'MA', 'MG', 'MS', 'MT', 'PA', 'PB', 'PE', 'PI', 'PR', 'RJ', 'RN', 'RO', 'RS', 'SC', 'SE', 'SP']


In [ ]:
datasets_clean = {
    'customers': customers_clean,
    'geolocation': geolocation_clean,
    'order_items': order_items_clean,
    'order_payments': order_payments_clean,
    'order_reviews': order_reviews_clean,
    'orders': orders_clean,
    'products': products_clean,
    'sellers': sellers_clean,
    'category_translation': category_translation_clean
}

In [ ]:
for name, df in datasets_clean.items():
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")

customers: 99,441 rows, 5 columns
geolocation: 738,332 rows, 5 columns
order_items: 112,650 rows, 7 columns
order_payments: 103,886 rows, 5 columns
order_reviews: 99,224 rows, 7 columns
orders: 99,441 rows, 8 columns
products: 32,951 rows, 10 columns
sellers: 3,095 rows, 4 columns
category_translation: 71 rows, 2 columns


In [ ]:
for name, df in datasets_clean.items():
    print(f"{name}: {df.duplicated().sum():,} exact duplicate rows")

customers: 0 exact duplicate rows
geolocation: 0 exact duplicate rows
order_items: 0 exact duplicate rows
order_payments: 0 exact duplicate rows
order_reviews: 0 exact duplicate rows
orders: 0 exact duplicate rows
products: 0 exact duplicate rows
sellers: 0 exact duplicate rows
category_translation: 0 exact duplicate rows


In [ ]:
for name, df in datasets_clean.items():
    missing = df.isnull().sum().sum()
    print(f"{name}: {missing:,} missing values")

customers: 0 missing values
geolocation: 0 missing values
order_items: 0 missing values
order_payments: 0 missing values
order_reviews: 145,903 missing values
orders: 4,908 missing values
products: 1,838 missing values
sellers: 0 missing values
category_translation: 0 missing values
